# 제주 특산물 가격 예측 — v7.0.0 순수 트리 회귀

| 항목 | 내용 |
|------|------|
| **버전** | v7.0.0 |
| **날짜** | 2026-03-17 |
| **모델** | CatBoost + LightGBM + XGBoost (VotingRegressor) |
| **검증** | 시즌 Hold-Out (2022-03) |
| **출력** | results/submission_v7.0.0.csv |

---

## 설계 원칙 — 지금까지 실험 결과 총정리

| 버전 | 모델 | Public | 교훈 |
|------|------|--------|------|
| v1.0.1 | DNN + 달력 7개 | **658.6** ✅ | 단순 달력 피처가 최강 |
| v4.0.0 | LGB+CB 트리 | 672.5 | DNN보다 소폭 낮음 |
| v4.1.0 | LGB+CB + EMA | 1170.5 ❌ | **EMA freeze → 절대 금지** |
| v5.0.0 | LGB+CB + 명절거리 | — | 명절 거리 피처 추가 |

### v7.0.0 채택 이유

1. **DNN 제거** — 단순할수록 점수가 좋았음. 트리만으로 충분
2. **EMA/lag 피처 없음** — 테스트 기간 freeze로 치명적 손해
3. **달력 7개 + 명절 거리 2개** — 점수를 올린 피처만 유지
4. **2그룹 (TG / 비TG)** — 가장 명확한 특성 분리
5. **Hold-Out 검증** — KFold는 미래 데이터 혼입으로 MAE 낙관적
6. **검증 후 전체 재학습** — 제출용 모델은 최대 데이터로

### 팀원별 반영 요소

| 팀원 | 반영 내용 |
|------|----------|
| **신우철** | 달력 7개 피처, year_month 누적 인코딩, TG sqrt/비TG log1p, 이상치 처리, Hold-Out 검증 |
| **김대원** | dist_seollal / dist_chuseok, 일요일 후처리 |
| **박효준** | EDA 확인된 이상치 임계값, 명절 패턴 반영 |

---
## 1. 라이브러리 로드

In [ ]:
import os
import pandas as pd
import numpy as np
import holidays
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import platform
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

try:
    from korean_font import set_korean_font
    set_korean_font()
except:
    plt.rcParams['font.family'] = 'Malgun Gothic' if platform.system() == 'Windows' else 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 2024
np.random.seed(SEED)

DATA_PATH = '../data/'
train = pd.read_csv(DATA_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATA_PATH + 'test.csv',  encoding='utf-8-sig')
sub   = pd.read_csv(DATA_PATH + 'sample_submission.csv', encoding='utf-8-sig')

train['timestamp'] = pd.to_datetime(train['timestamp'])
test['timestamp']  = pd.to_datetime(test['timestamp'])

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'학습 기간: {train["timestamp"].min().date()} ~ {train["timestamp"].max().date()}')
print(f'테스트 기간: {test["timestamp"].min().date()} ~ {test["timestamp"].max().date()}')

---
## 3. 이상치 처리

> 박효준 EDA + 신우철 v1.0.1 확립 임계값  
> 제거가 아닌 **품목 평균으로 대체** — 학습 데이터 손실 최소화

In [ ]:
### 2-4. 계절성 패턴 — 월별 평균 가격 (Heatmap)
df_nz = train[train['price(원/kg)'] > 0].copy()
df_nz['month'] = df_nz['timestamp'].dt.month
df_nz['year']  = df_nz['timestamp'].dt.year

pivot = df_nz.groupby(['item', 'month'])['price(원/kg)'].mean().unstack()
pivot = pivot.loc[ITEMS]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': '평균 가격 (원/kg)'})
ax.set_title('품목 × 월별 평균 가격 히트맵  (계절성 패턴 확인)', fontsize=13, fontweight='bold')
ax.set_xlabel('월')
ax.set_ylabel('품목')
ax.set_xticklabels([f'{m}월' for m in range(1, 13)])
plt.tight_layout()
plt.savefig('./results/eda_heatmap.png', dpi=150, bbox_inches='tight') if os.path.exists('./results') else None
plt.show()

print()
print('→ 달력(월) 피처가 가격의 지배적 신호임을 히트맵에서 확인')

---
## 4. 전처리 — 달력 피처 + 명절 거리

| 피처 | 출처 | 근거 |
|------|------|------|
| year / month / day / week_day | v1.0.1 기본 | 날짜 기본 정보 |
| year_month (누적 개월) | v1.0.1 핵심 | 시간 흐름 인식 |
| week_num (누적 주차) | v1.0.1 핵심 | 주 단위 패턴 |
| holiday | v1.0.1 | 공휴일 영향 |
| dist_seollal | v5.0.0 + 김대원 | 설날 접근 시 가격 급등 |
| dist_chuseok | v5.0.0 + 김대원 | 추석 접근 시 가격 급등 |

In [ ]:
### 2-2. 품목별 가격 분포 (Boxplot) + 이상치 임계값
ITEMS  = ['TG', 'RD', 'BC', 'CB', 'CR']
COLORS = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
OUTLIER_THR_VIZ = {'TG': 20000, 'RD': 5000, 'BC': 8000, 'CB': 2300}

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for ax, item, color in zip(axes, ITEMS, COLORS):
    data = train[train['item'] == item]['price(원/kg)']
    data_nz = data[data > 0]
    ax.boxplot(data_nz, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.7),
               medianprops=dict(color='black', lw=2),
               flierprops=dict(marker='o', markersize=2, alpha=0.4))
    if item in OUTLIER_THR_VIZ:
        ax.axhline(OUTLIER_THR_VIZ[item], color='red', lw=2, linestyle='--', label=f'임계값 {OUTLIER_THR_VIZ[item]:,}')
        ax.legend(fontsize=8)
    ax.set_title(f'{item}', fontsize=13, fontweight='bold')
    ax.set_ylabel('가격 (원/kg)' if ax == axes[0] else '')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('품목별 가격 분포 (빨간 점선 = 이상치 임계값)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/eda_boxplot.png', dpi=150, bbox_inches='tight') if os.path.exists('./results') else None
plt.show()

---
## 5. One-Hot 인코딩 및 피처 정의

---
## 2. EDA — 탐색적 데이터 분석

> 이상치 임계값 설정 및 피처 선택의 근거 확인

---
## 6. 시즌 Hold-Out 검증 (2022-03)

> 테스트 기간 = 2023-03 → 같은 조건: 2022-03을 검증 세트로  
> 검증 MAE × 1.26 ≈ 예상 Public Score

In [ ]:
OUTLIER_THR = {'TG': 20000, 'RD': 5000, 'BC': 8000, 'CB': 2300}

for item, thr in OUTLIER_THR.items():
    mask = (train['item'] == item) & (train['price(원/kg)'] > thr)
    if mask.any():
        mean_val = train[(train['item'] == item) & (train['price(원/kg)'] > 0)]['price(원/kg)'].mean()
        train.loc[mask, 'price(원/kg)'] = mean_val
        print(f'  {item}: {mask.sum()}개 → 평균 {mean_val:.0f}원으로 대체')

print('이상치 처리 완료')

---
## 3. 전처리 — 달력 피처 + 명절 거리

| 피처 | 출처 | 근거 |
|------|------|------|
| year / month / day / week_day | v1.0.1 기본 | 날짜 기본 정보 |
| year_month (누적 개월) | v1.0.1 핵심 | 시간 흐름 인식 |
| week_num (누적 주차) | v1.0.1 핵심 | 주 단위 패턴 |
| holiday | v1.0.1 | 공휴일 영향 |
| dist_seollal | v5.0.0 + 김대원 | 설날 접근 시 가격 급등 |
| dist_chuseok | v5.0.0 + 김대원 | 추석 접근 시 가격 급등 |

---
## 7. 검증 결과 시각화

---
## 4. One-Hot 인코딩 및 피처 정의

---
## 8. 전체 데이터 재학습 (제출용)

---
## 5. 시즌 Hold-Out 검증 (2022-03)

> 테스트 기간 = 2023-03 → 같은 조건: 2022-03을 검증 세트로  
> 검증 MAE × 1.26 ≈ 예상 Public Score

---
## 9. 테스트 예측 및 후처리

In [ ]:
# VotingRegressor (CatBoost + LightGBM + XGBoost)
def make_voter(seed=SEED):
    return VotingRegressor([
        ('cat',  CatBoostRegressor(
            iterations=2000, learning_rate=0.03, depth=6,
            random_seed=seed, verbose=0, loss_function='MAE'
        )),
        ('lgbm', LGBMRegressor(
            n_estimators=2000, learning_rate=0.03, num_leaves=63,
            random_state=seed, verbose=-1
        )),
        ('xgb',  XGBRegressor(
            n_estimators=2000, learning_rate=0.03, max_depth=6,
            random_state=seed, verbosity=0, objective='reg:absoluteerror'
        ))
    ])

print('Hold-Out 검증 학습 중...')
voter_ho = make_voter()
voter_ho.fit(X_tr, y_tr)

# 역변환 후 MAE
pred_vl_t = voter_ho.predict(X_vl)
pred_vl = np.where(
    is_tg_vl,
    np.power(np.clip(pred_vl_t, 0, None), 2),
    np.expm1(pred_vl_t)
)

mae_tg  = mean_absolute_error(y_vl_raw[is_tg_vl],  pred_vl[is_tg_vl])
mae_nt  = mean_absolute_error(y_vl_raw[~is_tg_vl], pred_vl[~is_tg_vl])
mae_all = mean_absolute_error(y_vl_raw, pred_vl)

print('=' * 55)
print('  Hold-Out 검증 결과 (2022-03)')
print('=' * 55)
print(f'  TG (감귤)  MAE : {mae_tg:>8,.2f} 원/kg')
print(f'  비TG       MAE : {mae_nt:>8,.2f} 원/kg')
print(f'  전체       MAE : {mae_all:>8,.2f} 원/kg')
print(f'  예상 Public ≈  {mae_all * 1.26:>8,.0f} 원')
print('=' * 55)

---
## 10. 제출 파일 생성

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 실제 vs 예측
max_v = max(y_vl_raw.max(), pred_vl.max())
axes[0].scatter(y_vl_raw, pred_vl, alpha=0.3, s=8,
                c=is_tg_vl.astype(int), cmap='coolwarm')
axes[0].plot([0, max_v], [0, max_v], 'k--', lw=1.5)
axes[0].set_title(f'실제 vs 예측  MAE={mae_all:.1f}', fontsize=12)
axes[0].set_xlabel('실제 가격'); axes[0].set_ylabel('예측 가격')
axes[0].grid(alpha=0.3)

# 잔차 분포
res = y_vl_raw.values - pred_vl
axes[1].hist(res, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', lw=2, linestyle='--')
axes[1].set_title(f'잔차 분포  mean={res.mean():.1f}', fontsize=12)
axes[1].set_xlabel('잔차'); axes[1].grid(alpha=0.3)

# 피처 중요도 (CatBoost 기준)
cat_model = voter_ho.estimators_[0]  # CatBoost
imp = pd.Series(cat_model.get_feature_importance(), index=FEAT_COLS).sort_values(ascending=False)
colors_imp = ['tomato' if 'dist_' in c else
              'mediumseagreen' if c in ['year_month','week_num','month','holiday'] else
              'steelblue' for c in imp.head(15).index]
imp.head(15).plot(kind='barh', ax=axes[2], color=colors_imp[::-1], alpha=0.85)
axes[2].invert_yaxis()
axes[2].set_title('CatBoost 피처 중요도 Top-15\n(빨강=명절거리, 초록=핵심 달력)', fontsize=11)
axes[2].grid(axis='x', alpha=0.3)

plt.suptitle('v7.0.0 Hold-Out 검증 결과', fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('./results', exist_ok=True)
plt.savefig('./results/eval_v7.0.0.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. 최종 결과 요약

In [ ]:
# 전체 타겟 변환
is_tg_all = Xy.get('item_TG', pd.Series([False]*len(Xy))).astype(bool)
y_all = np.where(is_tg_all, np.sqrt(Xy['price(원/kg)']), np.log1p(Xy['price(원/kg)']))
X_all = Xy[FEAT_COLS]

print('전체 데이터 재학습 중...')
voter_full = make_voter()
voter_full.fit(X_all, y_all)
print('재학습 완료')

---
## 8. 테스트 예측 및 후처리

In [ ]:
pred_test_t = voter_full.predict(X_test)

# 역변환
pred_test = np.where(
    is_tg_test,
    np.power(np.clip(pred_test_t, 0, None), 2),
    np.expm1(pred_test_t)
)
pred_test = np.clip(pred_test, 0, None)

# 후처리 1: 품목별 최솟값 미만 → 0
result_df = test_f[['ID','item','timestamp']].copy()
result_df['dow']    = result_df['timestamp'].dt.dayofweek
result_df['answer'] = pred_test

MIN_THR = {'TG': 400, 'CB': 50, 'RD': 10, 'CR': 150, 'BC': 100}
for it, val in MIN_THR.items():
    mask = (result_df['item'] == it) & (result_df['answer'] < val)
    result_df.loc[mask, 'answer'] = 0

# 후처리 2: 일요일 → 0 (경매 휴장)
result_df.loc[result_df['dow'] == 6, 'answer'] = 0

print('[품목별 예측 통계]')
print(result_df.groupby('item')['answer'].agg(['mean','min','max']).round(1))

---
## 9. 제출 파일 생성

In [ ]:
result = sub[['ID']].merge(result_df[['ID','answer']], on='ID', how='left')
result['answer'] = result['answer'].fillna(0)

SUBMISSION_PATH = './results/submission_v7.0.0.csv'
result.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')

print(f'저장 완료: {SUBMISSION_PATH}')
print(f'행 수: {len(result)},  결측: {result["answer"].isnull().sum()}')

---
## 10. 최종 결과 요약

In [ ]:
print('=' * 65)
print('  Jeju_ver7.0.0  순수 트리 회귀 최종 결과')
print('=' * 65)
print()
print('  [모델]  CatBoost + LightGBM + XGBoost  VotingRegressor')
print(f'  [피처]  {len(FEAT_COLS)}개 (달력 9개 + dist 2개 + OHE)')
print()
print('  [채택한 긍정 요소]')
print('  - 달력 피처 중심  (v1.0.1 Public 658.6 근거)')
print('  - year_month 누적 인코딩  (v1.0.1 핵심)')
print('  - dist_seollal / dist_chuseok  (v5.0.0 + 김대원)')
print('  - TG sqrt / 비TG log1p 변환  (v1.0.1)')
print('  - 시즌 Hold-Out 검증  (v5.0.0)')
print()
print('  [제거한 요소]')
print('  - EMA 피처  (freeze → v4.1.0 Public 1170.5 실패)')
print('  - DNN  (단순할수록 좋았던 실험 결과 반영)')
print()
print('  [Hold-Out 검증 MAE (2022-03)]')
print(f'  TG       {mae_tg:>8,.2f} 원/kg')
print(f'  비TG     {mae_nt:>8,.2f} 원/kg')
print(f'  전체     {mae_all:>8,.2f} 원/kg')
print(f'  예상 Public ≈ {mae_all * 1.26:,.0f}원')
print()
print(f'  제출 파일: {SUBMISSION_PATH}')
print('=' * 65)